In [1]:
# Imports
import sys
from pathlib import Path

# Resolve project root and ensure it's on sys.path
ROOT = Path.cwd().resolve()
for _ in range(5):
    if (ROOT / "pyproject.toml").exists() or (ROOT / "raw_data").exists():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils import discretize_preprocess

In [2]:
# Preprocess data
from pathlib import Path

dataset_path = ROOT / "raw_data" / "car.csv"
output_path = ROOT / "discretized_data" / "car.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

discretize_preprocess(str(dataset_path), str(output_path), bins=10, strategy='uniform')

Preprocessing: /home/adity/github/katabatic-mentorship-repo/raw_data/car.csv
Saved preprocessed discrete dataset to: /home/adity/github/katabatic-mentorship-repo/discretized_data/car.csv


In [3]:
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.models.arf_alex.adapter import KatabaticARF

# Set paths
input_csv = str(output_path)
output_dir = str(ROOT / "sample_data" / "car")
real_test_dir = output_dir
synthetic_dir = str(ROOT / "synthetic" / "car" / "arf")

# ARF hyperparameters
model_config = {
    "num_trees": 50,        # Number of trees in the forest
    "max_iters": 10,        # Adversarial iterations
    "min_node_size": 5,     # Minimum samples per leaf (controls overfitting)
    "delta": 0.0,           # Convergence tolerance
}

pipeline = TrainTestSplitPipeline(model=KatabaticARF)

pipeline.run(
    input_csv=input_csv,
    output_dir=output_dir,
    synthetic_dir=synthetic_dir,
    real_test_dir=real_test_dir,
    **model_config
)

Loaded data with shape: (1728, 7)
Saved train/test full data
Train size: (1382, 7), Test size: (346, 7)
Train label distribution:
 6
2    0.700434
0    0.222142
1    0.039797
3    0.037627
Name: proportion, dtype: float64
Test label distribution:
 6
2    0.699422
0    0.222543
1    0.040462
3    0.037572
Name: proportion, dtype: float64
Saved X/y split
Training shape: (1382, 6) (1382,)
Test shape: (346, 6) (346,)
Loading ARF data from: /home/adity/github/katabatic-mentorship-repo/sample_data/car
Training ARF on 1382 rows...
Generating synthetic data to: /home/adity/github/katabatic-mentorship-repo/synthetic/car/arf
Saved artifacts.


/home/adity/github/katabatic-mentorship-repo/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/adity/github/katabatic-mentorship-repo/.venv/lib/python3.11/site-packages/xgboost/training.py:199: UserWarning: [12:17:08] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: Results/car/arf_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.7052
F1 Score: 0.6249

MLP:
Accuracy: 0.8526
F1 Score: 0.8307

RF:
Accuracy: 0.8150
F1 Score: 0.8010

XGBoost:
Accuracy: 0.8064
F1 Score: 0.7992


'Train test split pipeline executed successfully.'